<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook subsets the All of Us short-read v7 VDS to the Phase 1 long-read cohort on Terra. Sample lists come from GATK-SV and long-read SV VCFs (~1,027 non-HPRC samples; ~989–990 overlapping GATK-SV). It writes subset VDS/MatrixTables and runs Hail `sample_qc`.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import pandas as pd
import numpy as np
import os
import re

In [ ]:
import pysam
from pysam import VariantFile

In [ ]:
from google.cloud import storage

In [ ]:
import hail as hl
from hail.plot import show
from pprint import pprint

## Define helper functions

In [ ]:
# True if a Hail MatrixTable already exists at gcs_path (checks for its README.txt).
def mt_exists(gcs_path):
    (gcs_bucket_name, gcs_obj) = re.split("\/", re.sub("gs://", "", gcs_path), maxsplit=1)
    
    storage_client = storage.Client()
    gcs_bucket = storage_client.bucket(gcs_bucket_name)
    stats = storage.Blob(bucket=gcs_bucket, name=f'{gcs_obj}/README.txt').exists(storage_client)
    
    return stats

In [ ]:
# True if a Hail VariantDataset already exists at gcs_path (checks its reference_data README.txt).
def vds_exists(gcs_path):
    (gcs_bucket_name, gcs_obj) = re.split("\/", re.sub("gs://", "", gcs_path), maxsplit=1)
    
    storage_client = storage.Client()
    gcs_bucket = storage_client.bucket(gcs_bucket_name)
    stats = storage.Blob(bucket=gcs_bucket, name=f'{gcs_obj}/reference_data/README.txt').exists(storage_client)
    
    return stats

In [ ]:
# Workspace details Terra sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']
workspace = os.environ['WORKSPACE_NAME']
namespace = os.environ['WORKSPACE_NAMESPACE']

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Short-read SV callsets (GATK-SV): the Phase I joint VCF, and the v7 callset subset to long-read samples.
# Each is downloaded once.
if not os.path.exists("AoU_srWGS_SV_PhaseI.vcf.gz"):
    !gsutil -m cp {AOU_LR_DATA}/aou-wgs-sv/phase1/joint-vcf/AoU_srWGS_SV_PhaseI.vcf.gz .
    !gsutil -m cp {AOU_LR_DATA}/aou-wgs-sv/phase1/joint-vcf/AoU_srWGS_SV_PhaseI.vcf.gz.tbi .

In [ ]:
if not os.path.exists("cohort_AoUSVPhaseII.v7.LRsamples.vcf.gz"):
    !gsutil cp {AOU_LR_DATA}/yulia/cohort_AoUSVPhaseII.v7.LRsamples.vcf.gz .

In [ ]:
#sr_sv_samples = !zgrep -m1 '^#CHROM' AoU_srWGS_SV_PhaseI.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
# Sample IDs in the short-read SV VCF header.
sr_sv_samples = !zgrep -m1 '^#CHROM' cohort_AoUSVPhaseII.v7.LRsamples.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
len(sr_sv_samples)

In [ ]:
# Long-read joint SV callsets at two calibration sensitivities (sens_09 and the stricter sens_07).
if not os.path.exists("concat_annotated.sens_09.vcf.gz"):
    !gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_09.vcf.gz .
        
if not os.path.exists("concat_annotated.sens_07.vcf.gz"):        
    !gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_07.vcf.gz .

In [ ]:
sv_sens_09_vcf = 'concat_annotated.sens_09.vcf.gz'
sv_sens_07_vcf = 'concat_annotated.sens_07.vcf.gz'

In [ ]:
# Peek at the first records of the sens_09 callset.
!cat {sv_sens_09_vcf} | zcat | head -n 2000 | grep -v '^#' | head -n 3 | cut -f1-9

In [ ]:
sv_sens_09_in = VariantFile(sv_sens_09_vcf)  # auto-detect input format

for i, rec in enumerate(sv_sens_09_in):
    print(f'{i} {rec.chrom} {rec.pos} {rec.info.values()}')
    
    if i > 10:
        break

## List long read samples

In [ ]:
# Sample IDs in the long-read SV callset (1,074 samples, including HPRC), also written to samples_1074.txt.
lr_sv_samples = !zgrep -m1 '^#CHROM' concat_annotated.sens_09.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
len(lr_sv_samples)

In [ ]:
!zgrep -m1 '^#CHROM' concat_annotated.sens_09.vcf.gz | cut -f10- | sed 's/\t/\n/g' > samples_1074.txt


In [ ]:
with open('samples_1074.txt', 'r') as file:
    sample_names = file.readlines()

sample_names = [name.strip() for name in sample_names]

In [ ]:
len(sample_names)

## List long read samples without HPRC samples

In [ ]:
# Drop the HPRC samples (IDs starting HG or NA), leaving the 1,027 AoU long-read samples.
common_samples_1027 = [element for element in lr_sv_samples if not (element.startswith('HG') or element.startswith('NA'))]
len(common_samples_1027)

## List long read samples with GATK-SV calls available

In [ ]:
#common_samples_989 = list(set(sr_sv_samples) & set(lr_sv_samples))
#len(common_samples_989)

In [ ]:
# AoU long-read samples that also have short-read SV calls (990).
common_samples_990 = list(set(sr_sv_samples) & set(lr_sv_samples))
len(common_samples_990)

## Initialize Hail

In [ ]:
# Start Hail with more driver and executor memory.
spark_conf_more_ram = dict()
spark_conf_more_ram["spark.executor.memory"] = "8g"
spark_conf_more_ram["spark.driver.memory"] = "196g"

# hl.init(default_reference='GRCh38', idempotent=True, spark_conf=spark_conf_more_ram)

hl.init(idempotent=True, spark_conf=spark_conf_more_ram)

In [ ]:
hl.default_reference('GRCh38')

## Subset v7 VDS to samples that have long reads

In [ ]:
# The short-read v7 SNV/indel callset (VDS).
vds = hl.vds.read_vds(f'{AOU_LR_DATA}/v7/wgs/with_aian_no_prod/vds/aou_srwgs_short_variants_v7_with_aian_no_prod.vds')

In [ ]:
# Subset the VDS to the 1,027 long-read samples. The subsetting code is commented out because the subset
# was already written; the next cell reads it.
if not vds_exists(f'{bucket}/scratch/srs-subset.1027.chr22.vds'):
    #callset_sample_filtered_1027 = hl.vds.filter_samples(vds, common_samples_1027, keep=True, remove_dead_alleles=True)
    #callset_sample_filtered_1027.write(f'{bucket}/scratch/srs-subset.1027.chr22.vds', overwrite=True)
    #callset_sample_filtered_1027.write(f'{bucket}/scratch/srs-subset.1027.vds', overwrite=True)
    
    pass

In [ ]:
callset_sample_filtered_1027 = hl.vds.read_vds(f'{bucket}/scratch/srs-subset.1027.chr22.vds')

## Subset v7 VDS to samples that have long reads and GATK-SV calls

In [ ]:
# Same for the 989-sample subset (already written).
if not vds_exists(f'{bucket}/scratch/srs-subset.989.vds'):
    #callset_sample_filtered_989 = hl.vds.filter_samples(callset_sample_filtered_1027, common_samples_989, keep=True, remove_dead_alleles=True)
    #callset_sample_filtered_989.write(f'{bucket}/scratch/srs-subset.989.vds', overwrite=True)
    
    pass

In [ ]:
callset_sample_filtered_989 = hl.vds.read_vds(f'{bucket}/scratch/srs-subset.989.vds')

In [ ]:
# Subset to the 990 samples with v7 short-read SV calls, if not already written.
if not vds_exists(f'{bucket}/scratch/srs-subset.990.vds'):
    callset_sample_filtered_990 = hl.vds.filter_samples(callset_sample_filtered_1027, common_samples_990, keep=True, remove_dead_alleles=True)
    callset_sample_filtered_990.write(f'{bucket}/scratch/srs-subset.990.vds', overwrite=True)

In [ ]:
callset_sample_filtered_990 = hl.vds.read_vds(f'{bucket}/scratch/srs-subset.990.vds')

## Check that we got the number of samples correct

In [ ]:
# Check the sample counts of each subset.
callset_sample_filtered_1027.n_samples()

In [ ]:
callset_sample_filtered_989.n_samples()

In [ ]:
callset_sample_filtered_990.n_samples()

## Densify subsetted VDS objects

In [ ]:
# Densify the 1,027-sample VDS into a MatrixTable: convert local-allele AD and GT to global alleles, fill
# in reference blocks, and add per-site allele counts and frequencies (info). ('or True' forces a rewrite.)
if not mt_exists(f'{bucket}/scratch/srs-subset.1027.mt') or True:
    mt_1027 = callset_sample_filtered_1027.variant_data.annotate_entries(
        AD = hl.vds.local_to_global(callset_sample_filtered_1027.variant_data.LAD, 
                                    callset_sample_filtered_1027.variant_data.LA, 
                                    n_alleles = hl.len(callset_sample_filtered_1027.variant_data.alleles), 
                                    fill_value = 0, 
                                    number = 'R')
    )
    
    mt_1027 = mt_1027.annotate_entries(GT = hl.vds.lgt_to_gt(mt_1027.LGT, mt_1027.LA))
    mt_1027 = hl.vds.to_dense_mt(hl.vds.VariantDataset(callset_sample_filtered_1027.reference_data, mt_1027))
    mt_1027 = mt_1027.annotate_rows(info = hl.agg.call_stats(mt_1027.GT, mt_1027.alleles))
    mt_1027.write(f'{bucket}/scratch/srs-subset.1027.mt', overwrite=True)

In [ ]:
mt_1027 = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.1027.mt')

In [ ]:
# Hail per-sample QC metrics.
mt_qc_1027 = hl.sample_qc(mt_1027)

In [ ]:
mt_qc_1027.describe()

In [ ]:
mt_qc_1027.cols().show()

In [ ]:
# Same for the 989-sample subset.
if not mt_exists(f'{bucket}/scratch/srs-subset.989.mt') or True:
    mt_989 = callset_sample_filtered_989.variant_data.annotate_entries(
        AD = hl.vds.local_to_global(callset_sample_filtered_989.variant_data.LAD, 
                                    callset_sample_filtered_989.variant_data.LA, 
                                    n_alleles = hl.len(callset_sample_filtered_989.variant_data.alleles), 
                                    fill_value = 0, 
                                    number = 'R')
    )
    
    mt_989 = mt_989.annotate_entries(GT = hl.vds.lgt_to_gt(mt_989.LGT, mt_989.LA))
    mt_989 = hl.vds.to_dense_mt(hl.vds.VariantDataset(callset_sample_filtered_989.reference_data, mt_989))
    mt_989 = mt_989.annotate_rows(info = hl.agg.call_stats(mt_989.GT, mt_989.alleles))
    mt_989.write(f'{bucket}/scratch/srs-subset.989.mt', overwrite=True)

In [ ]:
mt_989 = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.989.mt')

In [ ]:
mt_qc_989 = hl.sample_qc(mt_989)

In [ ]:
mt_qc_989.describe()

In [ ]:
mt_qc_989.cols().show()

In [ ]:
# Same for the 990-sample subset.
if not mt_exists(f'{bucket}/scratch/srs-subset.990.mt') or True:
    mt_990 = callset_sample_filtered_990.variant_data.annotate_entries(
        AD = hl.vds.local_to_global(callset_sample_filtered_990.variant_data.LAD, 
                                    callset_sample_filtered_990.variant_data.LA, 
                                    n_alleles = hl.len(callset_sample_filtered_990.variant_data.alleles), 
                                    fill_value = 0, 
                                    number = 'R')
    )
    
    mt_990 = mt_990.annotate_entries(GT = hl.vds.lgt_to_gt(mt_990.LGT, mt_990.LA))
    mt_990 = hl.vds.to_dense_mt(hl.vds.VariantDataset(callset_sample_filtered_990.reference_data, mt_990))
    mt_990 = mt_990.annotate_rows(info = hl.agg.call_stats(mt_990.GT, mt_990.alleles))
    mt_990.write(f'{bucket}/scratch/srs-subset.990.mt', overwrite=True)

In [ ]:
mt_990 = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.990.mt')

In [ ]:
mt_qc_990 = hl.sample_qc(mt_990)

In [ ]:
mt_qc_990.describe()

In [ ]:
# Distribution across samples of the Ti/Tv ratio and the het/hom-alt ratio.
mt_qc_990.aggregate_cols(hl.agg.stats(mt_qc_990.sample_qc.r_ti_tv))

In [ ]:
mt_qc_990.aggregate_cols(hl.agg.stats(mt_qc_990.sample_qc.r_het_hom_var))